This script fetches historical hourly and daily weather data (temperature, precipitation, snow, wind, humidity, weather code) from the Open-Meteo Archive API for multiple monitoring stations (2021–2025). Station metadata is read from a CSV file. Results are saved as separate hourly and daily CSVs.

Developed with the assistance of [Gemini](https://gemini.google.com)

In [1]:
import openmeteo_requests
import polars as pl
import requests_cache
import os
from retry_requests import retry
from datetime import datetime, timedelta, timezone

BASE_DIR   = "../../data"

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Get location information
locations_df = pl.read_csv(os.path.join(BASE_DIR, "BASt", "AggregatedDataForKiel", "meta_data_used_stations.csv"))

location_Zst = locations_df["Zst"].to_list()
location_names = locations_df["name"].to_list()
latitudes = locations_df["lat"].to_list()
longitudes = locations_df["lon"].to_list()

# URL and parameters remain the same
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude":latitudes,
    "longitude": longitudes,
    "start_date": "2021-01-01",
    "end_date": "2025-12-31",
    "daily": ["weather_code", "temperature_2m_mean", "precipitation_sum"],
    "hourly": ["temperature_2m", "precipitation", "snow_depth", "snowfall", "wind_speed_10m", "wind_gusts_10m", "relative_humidity_2m", "weather_code"],
}
responses = openmeteo.weather_api(url, params=params)

In [4]:
# Hourly Data
BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "Weather", "weather_hourly.csv")

all_dataframes = []
# Process first location
for i, response in enumerate(responses):
    # Get name
    zst = location_Zst[i]
    name = location_names[i]
    
    # --- Process Hourly Data ---
    hourly = response.Hourly()
    
    #  calculate the range using datetime objects
    start_h = datetime.fromtimestamp(hourly.Time(), tz=timezone.utc)
    end_h = datetime.fromtimestamp(hourly.TimeEnd(), tz=timezone.utc)
    
    interval_h = timedelta(seconds=hourly.Interval())
    
    hourly_dataframe = pl.DataFrame({
        "date": pl.datetime_range(start_h, end_h - interval_h, interval_h, eager=True),
        "location_Zst": zst,
        "location_name": name,
        "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
        "precipitation": hourly.Variables(1).ValuesAsNumpy(),
        "snow_depth": hourly.Variables(2).ValuesAsNumpy(),
        "snowfall": hourly.Variables(3).ValuesAsNumpy(),
        "wind_speed_10m": hourly.Variables(4).ValuesAsNumpy(),
        "wind_gusts_10m": hourly.Variables(5).ValuesAsNumpy(),
        "relative_humidity_2m": hourly.Variables(6).ValuesAsNumpy(),
        "weather_code": hourly.Variables(7).ValuesAsNumpy(),
    })
    all_dataframes.append(hourly_dataframe)

final_dataframe = pl.concat(all_dataframes)

final_dataframe.write_csv(
    OUTPUT_DIR,
    datetime_format="%d.%m.%Y %H:%M"
)

In [5]:
# Daily Data
BASE_DIR   = "../../data"
OUTPUT_DIR = os.path.join(BASE_DIR, "Weather", "weather_daily.csv")

# --- Process Daily Data ---
daily = response.Daily()

start_d = datetime.fromtimestamp(daily.Time(), tz=timezone.utc)
end_d = datetime.fromtimestamp(daily.TimeEnd(), tz=timezone.utc)
interval_d = timedelta(seconds=daily.Interval())

daily_dataframe = pl.DataFrame({
    "date": pl.datetime_range(start_d, end_d - interval_d, interval_d, eager=True),
    "weather_code": daily.Variables(0).ValuesAsNumpy(),
    "temperature_2m_mean": daily.Variables(1).ValuesAsNumpy(),
    "precipitation_sum": daily.Variables(2).ValuesAsNumpy(),
})

daily_dataframe.write_csv(
    OUTPUT_DIR,
    datetime_format="%d.%m.%Y %H:%M"
)